# MiniMax H3｜ComfyUI 基础版（Colab）

在 Colab 上装好 ComfyUI 与 MiniMax H3 模型组，**不带任何工作流**——打开界面后自己拉节点、或拖入自己的工作流学习。
由上往下依序运行。**运行阶段请选 L4 GPU**（A100 也可以但很耗点数；T4 跑不动）。

| 步骤 | 做什么 | 时间（L4 实测） |
|---|---|---|
| ① 检查＋安装 | GPU／驱动不符就停下，不安装；通过才装 ComfyUI＋节点管理器＋ListHelper | 约 1 分钟 |
| ② 启动 | 背景启动 ComfyUI | 约 1 分钟 |
| ③ 打开界面 | Colab 内置代理（推荐）；③-b 是 cloudflare 备用 | 数秒 |
| ④ 下载模型 | 约 37 GB（ComfyUI 确定正常才下载） | 约 1 分钟 |

想直接做「会说话的数字人」：改开同一个 repo 的 `H3_Digital_Human.ipynb`。

**Hugging Face 令牌（建议）**：左侧栏「🔑 密钥」添加 `HF_TOKEN`（请用**只读**令牌），并打开「笔记本访问权」。

模型：`Minimax-h3_Singularity_x_10Eros-Max_PRISM`（int8，加速已融合，6 步）＋官方 Qwen3-VL 文本编码器与 VAE。授权依照各来源模型。

最新版：<https://github.com/dseditor/minimax-h3-colab>

In [ ]:
#@title ① 检查环境＋安装 ComfyUI（GPU 或驱动不符就停下，不会安装）
FIX_SRC = "# Colab 内置代理会把网址里的 %2F 解成 /，ComfyUI 的 /userdata/{file} 就配不到 → 工作流清单点开 404。\n# 这个小节点补一条「被拆成两段」的路由，读写同一个用户文件夹。\nimport os\nfrom aiohttp import web\nimport folder_paths\nfrom server import PromptServer\n\nroutes = PromptServer.instance.routes\nBASE = os.path.abspath(os.path.join(folder_paths.get_user_directory(), 'default'))\n\n\ndef _path(req):\n    p = os.path.abspath(os.path.join(BASE, req.match_info['sub'], req.match_info['file']))\n    if not p.startswith(BASE + os.sep):\n        raise web.HTTPForbidden()\n    return p\n\n\n@routes.get('/userdata/{sub}/{file}')\nasync def _get(req):\n    p = _path(req)\n    if not os.path.isfile(p):\n        return web.Response(status=404)\n    return web.FileResponse(p)\n\n\n@routes.post('/userdata/{sub}/{file}')\nasync def _post(req):\n    p = _path(req)\n    if os.path.exists(p) and req.query.get('overwrite', 'true') == 'false':\n        return web.Response(status=409, text='File already exists')\n    os.makedirs(os.path.dirname(p), exist_ok=True)\n    with open(p, 'wb') as fh:\n        fh.write(await req.read())\n    rel = os.path.relpath(p, BASE).replace(os.sep, '/')\n    if req.query.get('full_info', 'false') == 'true':\n        st = os.stat(p)\n        return web.json_response({'path': rel, 'size': st.st_size, 'modified': st.st_mtime})\n    return web.json_response(rel)\n\n\n@routes.delete('/userdata/{sub}/{file}')\nasync def _delete(req):\n    p = _path(req)\n    if not os.path.isfile(p):\n        return web.Response(status=404)\n    os.remove(p)\n    return web.Response(status=204)\n\n\nNODE_CLASS_MAPPINGS = {}\n"
import torch, shutil, subprocess, sys, os
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError('没有 GPU：请到「运行阶段 → 变更运行阶段类型」选 L4 GPU，再重跑这格')
cap = torch.cuda.get_device_capability(0)
drv = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', torch.cuda.get_device_name(0), '| capability', cap, '| VRAM', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('驱动:', drv, '| 目前 PyTorch CUDA:', torch.version.cuda, '| 可用磁盘', round(shutil.disk_usage('/content').free / 2**30, 1), 'GiB')
if cap < (8, 0):
    raise RuntimeError(f'这张卡（capability {cap}）跑不动 int8 模型，没有安装任何东西。请换 L4（或 A100）再重跑。')
if int(drv.split('.')[0]) < 580:
    raise RuntimeError(f'驱动 {drv} 不支持 CUDA 13（需要 580 以上），没有安装任何东西。请换一台运行阶段（L4）再试。')
print('✅ 环境检查通过，开始安装')
# HF 令牌先放进环境变量：之后启动的 ComfyUI（含 ListHelper 的模型下载节点）都会沿用
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('已读到 HF_TOKEN')
except Exception:
    print('没有设置 HF_TOKEN，之后改用匿名下载（人多时可能被限速）')
COMFY = Path('/content/ComfyUI')
if not COMFY.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI.git', str(COMFY)], check=True)
if not (torch.version.cuda or '').startswith('13'):
    print('安装 CUDA 13 版 PyTorch（int8 ConvRot 权重需要）…')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio',
                    '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt'), 'huggingface_hub', 'hf_xet'], check=True)
# 预装 ListHelper 节点组（含 ModelDownloader：在界面里从 Hugging Face 抓模型）
LH = COMFY / 'custom_nodes' / 'ComfyUI-ListHelper'
if not LH.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/dseditor/ComfyUI-ListHelper.git', str(LH)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(LH / 'requirements.txt')], check=True)
# Colab 内置代理（③）会把网址的 %2F 解开，工作流清单点开会 404 → 装一个补路由的小节点
FIX = COMFY / 'custom_nodes' / 'colab_userdata_slash_fix'
FIX.mkdir(parents=True, exist_ok=True)
(FIX / '__init__.py').write_text(FIX_SRC, encoding='utf-8')
# 节点管理器（ComfyUI-Manager 已并入官方，用 pip 套件＋启动旗标 --enable-manager）
if (COMFY / 'manager_requirements.txt').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'manager_requirements.txt')], check=True)
out = subprocess.run([sys.executable, '-c', 'import torch;print(torch.__version__, torch.version.cuda, torch.cuda.is_available())'],
                     capture_output=True, text=True)
print('ComfyUI 会用的 PyTorch：', out.stdout.strip(), out.stderr[-300:])


In [ ]:
#@title ② 启动 ComfyUI（背景运行）
import subprocess, time, urllib.request, sys
# --enable-cors-header：经 cloudflare 通道打开界面时，浏览器的 Origin（trycloudflare 网址）
# 跟转进来的 Host（127.0.0.1）对不上，ComfyUI 默认会回 403「访问遭拒」；开这个旗标才放行。
if 'server' in globals() and server.poll() is None:
    server.terminate(); server.wait(); print('已关闭旧的 ComfyUI，重新启动')
# --enable-compress-response-body：接口要抓一大包节点清单和前端文件，压缩后走通道快很多。
# 节点管理器安全等级：normal- ＝可从清单装节点，也可用 Git 网址安装（本机模式下）。
# 这台 Colab 用完即丢；但 HF_TOKEN 在环境变量里，自定义节点读得到 → HF 令牌请用「只读」的。
cfg = COMFY / 'user' / '__manager' / 'config.ini'
cfg.parent.mkdir(parents=True, exist_ok=True)
if not cfg.exists():
    cfg.write_text('[default]\nsecurity_level = normal-\n', encoding='utf-8')
log = open('/content/comfyui.log', 'w')
server = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--enable-cors-header', '--enable-manager', '--enable-compress-response-body'],
                          cwd=str(COMFY), stdout=log, stderr=subprocess.STDOUT)
for i in range(180):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        print('ComfyUI 已启动'); break
    except Exception:
        time.sleep(2)
else:
    print(open('/content/comfyui.log').read()[-3000:])
    raise RuntimeError('ComfyUI 没有启动成功，看上面的纪录')


In [ ]:
#@title ③ 打开 ComfyUI 界面（推荐：Colab 内置代理，快、稳、网址不外流）
# 运行后下面会出现一个链接；显示成 localhost 也没关系，直接点它就会开新标签页。
# ⚠️ 打开标签页的浏览器必须登录「跟这本 Colab 同一个 Google 账号」；账号不同就改用下一格 ③-b。
from google.colab import output
output.serve_kernel_port_as_window(8188)


In [ ]:
#@title ③-b（备用）用 cloudflare 网址打开界面：浏览器登录的 Google 账号和 Colab 不同时用这格
# 会产生一个 trycloudflare 网址，点开就是 ComfyUI；也是交给桌面版 AI 操作时要给的网址。
# ⚠️ 网址没有密码，拿到的人都能用你的 GPU —— 不要贴到公开的地方。
import subprocess, re, time, json, urllib.request
if not Path('/content/cloudflared').exists():
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/content/cloudflared'], check=True)
# 通道的输出写进文件，不要接 PIPE：PIPE 没人读会塞满，通道就卡住，网页停在加载画面
if 'tun' in globals() and tun.poll() is None:
    tun.terminate(); tun.wait()
open('/content/cloudflared.log', 'w').close()
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8188'],
                       stdout=open('/content/cloudflared.log', 'a'), stderr=subprocess.STDOUT)
url = None
for i in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
    if m:
        url = m.group(0); break
print('👉 ComfyUI 界面：', url or '（没拿到网址，看 /content/cloudflared.log）')


In [ ]:
#@title ④ 下载模型（约 37 GB）
# 默认直接从 Hugging Face 下载（最快）。HF 令牌已在 ① 读入。
# 只有在 Hugging Face 撞到流量上限时，才勾选 USE_DRIVE_CACHE：
#   勾选后会从「我的Google Drive/H3_models」拷贝（比 HF 慢），Google Drive没有的文件才去 HF 抓并顺便存一份。
USE_DRIVE_CACHE = False  #@param {type:"boolean"}
from huggingface_hub import hf_hub_download
import shutil, os
CACHE = None
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = Path('/content/drive/MyDrive/H3_models'); CACHE.mkdir(parents=True, exist_ok=True)
files = [
    ('OliviaRossi/Minimax-h3_Singularity-x-10Eros-Max_PRISM', 'Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors', 'diffusion_models'),
    ('Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'text_encoders'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_int8_convrot.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
]
for repo, name, folder in files:
    target = COMFY / 'models' / folder / Path(name).name
    target.parent.mkdir(parents=True, exist_ok=True)
    cached = CACHE / Path(name).name if CACHE else None
    if not target.is_file():
        if cached and cached.is_file():
            print('从Google Drive拷贝', cached.name); shutil.copy(str(cached), str(target))
        else:
            got = Path(hf_hub_download(repo_id=repo, filename=name, local_dir='/content/hf'))
            shutil.move(str(got), str(target))
    if cached and not cached.is_file():
        print('存一份到Google Drive', cached.name); shutil.copy(str(target), str(cached))
    print(folder, target.name, f'{target.stat().st_size / 2**30:.1f} GiB')


In [ ]:
#@title 🔍 出问题时跑这格：看 GPU 与 ComfyUI 纪录
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
log = open('/content/comfyui.log', encoding='utf-8', errors='ignore').read()
keys = ('error', 'Error', 'int8', 'quant', 'fallback', 'out of memory', 'loaded', 'Prompt executed')
print('\n'.join(l for l in log.splitlines() if any(k in l for k in keys))[-4000:])
